# Bank Marketing Term Deposit Prediction

This notebook is organized as a reusable project scaffold. It performs schema checks and exploratory analysis, then defines preprocessing, expert model templates, and a clustered expert classifier. Model training is intentionally disabled in this version.


## Notebook Contract

- Imports and configuration live at the top.
- EDA is allowed to run because it does not fit predictive models.
- Model objects are instantiated but not trained.
- Any cell that would fit, tune, cross-validate, or predict with a model is guarded by `RUN_TRAINING = False`.


In [ ]:
# Core imports
import os
import warnings
from pathlib import Path
from typing import Dict, Mapping, Optional, Sequence, Tuple

# Scientific Python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display helpers
from IPython.display import display

# Scikit-learn base utilities
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, PowerTransformer, RobustScaler
from sklearn.utils.validation import check_is_fitted

# Optional expert model libraries. The registry below will skip unavailable packages.
try:
    import xgboost as xgb
except ImportError:
    xgb = None

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None

try:
    import tabpfn as tp
except ImportError:
    tp = None

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)


In [ ]:
# Project configuration
RANDOM_STATE = 42
RUN_TRAINING = False
PRIMARY_SCORING = "balanced_accuracy"
N_CV_SPLITS = 5
N_MODEL_CLUSTERS = 3
MIN_CLUSTER_SIZE = 150

DATA_DIR = Path(".")
TRAIN_FEATURES_PATH = DATA_DIR / "bank_X_train.csv"
TRAIN_TARGET_PATH = DATA_DIR / "bank_y_train.csv"
TEST_FEATURES_PATH = DATA_DIR / "bank_X_test.csv"
TARGET_COLUMN = "y"

# The original target is coded as 1/2. The modeling target uses 0/1.
TARGET_TO_BINARY = {1: 0, 2: 1}
BINARY_TO_TARGET = {0: 1, 1: 2}
POSITIVE_CLASS = 1

EXPECTED_FEATURE_COLUMNS = [
    "age",
    "job",
    "marital",
    "education",
    "default",
    "balance",
    "housing",
    "loan",
    "contact",
    "day",
    "month",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome",
]

RAW_NUMERIC_FEATURES = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
RAW_CATEGORICAL_FEATURES = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
YES_NO_FEATURES = ["default", "housing", "loan"]

MONTH_TO_NUMBER = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "oct": 10,
    "nov": 11,
    "dec": 12,
}

ENGINEERED_NUMERIC_FEATURES = [
    "age",
    "balance",
    "balance_signed_log1p",
    "day",
    "day_sin",
    "day_cos",
    "duration",
    "duration_log1p",
    "campaign",
    "campaign_log1p",
    "pdays_clean",
    "previous",
    "month_number",
    "month_sin",
    "month_cos",
]
ENGINEERED_BINARY_FEATURES = ["was_contacted_before"]
ENGINEERED_CATEGORICAL_FEATURES = RAW_CATEGORICAL_FEATURES.copy()


## Data Loading and Validation

The raw CSV files are kept unchanged. Validation catches column drift, target encoding issues, and train/target row mismatches before any EDA or modeling code depends on them.


In [ ]:
def load_csv_data(
    train_features_path: Path = TRAIN_FEATURES_PATH,
    train_target_path: Path = TRAIN_TARGET_PATH,
    test_features_path: Path = TEST_FEATURES_PATH,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the project CSV files."""
    missing_paths = [path for path in [train_features_path, train_target_path, test_features_path] if not path.exists()]
    if missing_paths:
        missing_text = ", ".join(str(path) for path in missing_paths)
        raise FileNotFoundError(f"Missing required data file(s): {missing_text}")

    X_train_raw = pd.read_csv(train_features_path)
    y_train_raw = pd.read_csv(train_target_path)
    X_test_raw = pd.read_csv(test_features_path)
    return X_train_raw, y_train_raw, X_test_raw


def validate_schema(X_train: pd.DataFrame, y_train: pd.DataFrame, X_test: pd.DataFrame) -> pd.DataFrame:
    """Validate expected columns and row alignment, returning a compact schema report."""
    errors = []

    for split_name, frame in {"train": X_train, "test": X_test}.items():
        missing = [column for column in EXPECTED_FEATURE_COLUMNS if column not in frame.columns]
        extra = [column for column in frame.columns if column not in EXPECTED_FEATURE_COLUMNS]
        if missing:
            errors.append(f"{split_name} features are missing columns: {missing}")
        if extra:
            errors.append(f"{split_name} features contain unexpected columns: {extra}")

    if list(X_train.columns) != list(X_test.columns):
        errors.append("Train and test feature columns are not in the same order.")

    if TARGET_COLUMN not in y_train.columns or y_train.shape[1] != 1:
        errors.append(f"Target file must contain exactly one column named {TARGET_COLUMN!r}.")

    if len(X_train) != len(y_train):
        errors.append("Training features and target have different row counts.")

    if errors:
        raise ValueError("\n".join(errors))

    return pd.DataFrame(
        [
            {"frame": "X_train", "rows": len(X_train), "columns": X_train.shape[1]},
            {"frame": "y_train", "rows": len(y_train), "columns": y_train.shape[1]},
            {"frame": "X_test", "rows": len(X_test), "columns": X_test.shape[1]},
        ]
    )


def coerce_feature_types(frame: pd.DataFrame) -> pd.DataFrame:
    """Coerce known columns to stable notebook-friendly dtypes without changing semantics."""
    output = frame.copy()

    for column in RAW_NUMERIC_FEATURES:
        output[column] = pd.to_numeric(output[column], errors="coerce")

    for column in RAW_CATEGORICAL_FEATURES:
        normalized = output[column].where(output[column].isna(), output[column].astype(str).str.strip().str.lower())
        output[column] = normalized.astype("object")

    return output


def prepare_target(target_frame: pd.DataFrame) -> pd.Series:
    """Map the original 1/2 target labels to a binary 0/1 series."""
    raw_target = pd.to_numeric(target_frame[TARGET_COLUMN], errors="raise")
    observed_labels = set(raw_target.dropna().unique())

    if observed_labels <= set(TARGET_TO_BINARY):
        target = raw_target.map(TARGET_TO_BINARY)
    elif observed_labels <= {0, 1}:
        target = raw_target
    else:
        raise ValueError(f"Unexpected target labels: {sorted(observed_labels)}")

    if target.isna().any():
        raise ValueError("Target contains missing values after encoding.")

    return target.astype(int).rename("target")


def prepare_modeling_frames(
    X_train_raw: pd.DataFrame,
    y_train_raw: pd.DataFrame,
    X_test_raw: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """Return type-stable feature frames and binary target."""
    X_train = coerce_feature_types(X_train_raw)
    X_test = coerce_feature_types(X_test_raw)
    y_train = prepare_target(y_train_raw)
    return X_train, y_train, X_test


In [ ]:
X_train_raw, y_train_raw, X_test_raw = load_csv_data()
schema_report = validate_schema(X_train_raw, y_train_raw, X_test_raw)
X_train, y_train, X_test = prepare_modeling_frames(X_train_raw, y_train_raw, X_test_raw)
combined_features = pd.concat([X_train, X_test], ignore_index=True)

display(schema_report)
display(X_train.head())


## EDA Helpers

The functions below keep the notebook cells short and repeatable. Target-aware EDA only uses the training split; train plus test are combined only for schema, missingness, and distribution checks that do not inspect labels.


In [ ]:
def summarize_frames(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame) -> pd.DataFrame:
    """Summarize row counts, column counts, and duplicate rows."""
    return pd.DataFrame(
        [
            {"frame": "X_train", "rows": len(X_train), "columns": X_train.shape[1], "duplicate_rows": X_train.duplicated().sum()},
            {"frame": "y_train", "rows": len(y_train), "columns": 1, "duplicate_rows": np.nan},
            {"frame": "X_test", "rows": len(X_test), "columns": X_test.shape[1], "duplicate_rows": X_test.duplicated().sum()},
        ]
    )


def target_balance_table(y: pd.Series) -> pd.DataFrame:
    """Return target counts and rates for the binary modeling target."""
    counts = y.value_counts().sort_index()
    return pd.DataFrame(
        {
            "target": counts.index,
            "original_label": [BINARY_TO_TARGET.get(int(label), label) for label in counts.index],
            "count": counts.values,
            "rate": counts.values / len(y),
        }
    )


def missingness_table(frames: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    """Compare missingness by column and split."""
    rows = []
    for split_name, frame in frames.items():
        for column in frame.columns:
            missing_count = int(frame[column].isna().sum())
            rows.append(
                {
                    "split": split_name,
                    "column": column,
                    "missing_count": missing_count,
                    "missing_pct": missing_count / len(frame),
                    "dtype": str(frame[column].dtype),
                    "unique_values": int(frame[column].nunique(dropna=True)),
                }
            )
    return pd.DataFrame(rows).sort_values(["missing_pct", "missing_count"], ascending=False)


def numeric_profile(frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    """Profile numeric columns using robust quantiles and skew."""
    numeric = frame.loc[:, columns].apply(lambda series: pd.to_numeric(series, errors="coerce"))
    profile = numeric.agg(["count", "mean", "std", "min", "median", "max"]).T
    profile["missing_pct"] = numeric.isna().mean()
    profile["p01"] = numeric.quantile(0.01)
    profile["p25"] = numeric.quantile(0.25)
    profile["p75"] = numeric.quantile(0.75)
    profile["p99"] = numeric.quantile(0.99)
    profile["skew"] = numeric.skew(numeric_only=True)
    ordered_columns = ["count", "missing_pct", "mean", "std", "min", "p01", "p25", "median", "p75", "p99", "max", "skew"]
    return profile.loc[:, ordered_columns].sort_values("missing_pct", ascending=False)


def categorical_profile(frame: pd.DataFrame, columns: Sequence[str], top_n: int = 5) -> pd.DataFrame:
    """Profile categorical columns with cardinality and top values."""
    rows = []
    for column in columns:
        series = frame[column].astype("object")
        top_values = series.value_counts(dropna=False).head(top_n)
        rows.append(
            {
                "column": column,
                "missing_pct": series.isna().mean(),
                "unique_values": series.nunique(dropna=True),
                "top_values": "; ".join(f"{index}: {count}" for index, count in top_values.items()),
            }
        )
    return pd.DataFrame(rows).sort_values(["missing_pct", "unique_values"], ascending=False)


def add_bank_features(frame: pd.DataFrame, month_map: Mapping[str, int] = MONTH_TO_NUMBER) -> pd.DataFrame:
    """Add semantic and cyclic features without mutating the input frame."""
    output = coerce_feature_types(frame)

    pdays = pd.to_numeric(output["pdays"], errors="coerce")
    day = pd.to_numeric(output["day"], errors="coerce")
    month_number = output["month"].map(month_map)
    duration = pd.to_numeric(output["duration"], errors="coerce")
    campaign = pd.to_numeric(output["campaign"], errors="coerce")
    balance = pd.to_numeric(output["balance"], errors="coerce")

    output["was_contacted_before"] = pdays.ge(0).fillna(False).astype(int)
    output["pdays_clean"] = pdays.mask(pdays < 0, np.nan)
    output["month_number"] = month_number
    output["month_sin"] = np.sin(2 * np.pi * month_number / 12)
    output["month_cos"] = np.cos(2 * np.pi * month_number / 12)
    output["day_sin"] = np.sin(2 * np.pi * day / 31)
    output["day_cos"] = np.cos(2 * np.pi * day / 31)
    output["duration_log1p"] = np.log1p(duration.clip(lower=0))
    output["campaign_log1p"] = np.log1p(campaign.clip(lower=0))
    output["balance_signed_log1p"] = np.sign(balance) * np.log1p(balance.abs())

    return output


def target_rate_by_category(X: pd.DataFrame, y: pd.Series, column: str, min_count: int = 50) -> pd.DataFrame:
    """Compute positive-class rate by category for one column."""
    work = pd.DataFrame({column: X[column].fillna("<missing>").astype(str), "target": y.to_numpy()})
    rates = (
        work.groupby(column, dropna=False)["target"]
        .agg(row_count="size", positive_rate="mean")
        .reset_index()
        .query("row_count >= @min_count")
        .sort_values("positive_rate", ascending=False)
    )
    return rates


In [ ]:
def plot_target_balance(y: pd.Series) -> None:
    """Plot class balance for the binary training target."""
    balance = target_balance_table(y)
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=balance, x="target", y="count", hue="target", palette="Set2", legend=False, ax=ax)
    ax.set_title("Training Target Balance")
    ax.set_xlabel("Binary target")
    ax.set_ylabel("Rows")
    for index, row in balance.iterrows():
        ax.text(index, row["count"], f"{row['rate']:.1%}", ha="center", va="bottom")
    plt.tight_layout()
    plt.show()


def plot_missingness(frame: pd.DataFrame, max_rows: int = 4000) -> None:
    """Plot a sampled missingness map for columns that contain missing values."""
    missing_columns = frame.columns[frame.isna().any()].tolist()
    if not missing_columns:
        print("No missing values found in the provided frame.")
        return

    sample = frame.loc[:, missing_columns]
    if len(sample) > max_rows:
        sample = sample.sample(max_rows, random_state=RANDOM_STATE).sort_index()

    fig, ax = plt.subplots(figsize=(12, max(3, 0.45 * len(missing_columns))))
    ax.imshow(sample.isna().T, aspect="auto", interpolation="nearest", cmap="Greys")
    ax.set_yticks(range(len(missing_columns)))
    ax.set_yticklabels(missing_columns)
    ax.set_xlabel("Sampled row index")
    ax.set_title("Missingness Pattern")
    plt.tight_layout()
    plt.show()


def plot_numeric_distributions(X: pd.DataFrame, y: pd.Series, columns: Sequence[str], sample_size: int = 7000) -> None:
    """Plot numeric distributions by target class on a reproducible sample."""
    engineered = add_bank_features(X)
    available_columns = [column for column in columns if column in engineered.columns]
    plot_frame = engineered.loc[:, available_columns].copy()
    plot_frame["target"] = y.to_numpy()

    if len(plot_frame) > sample_size:
        plot_frame = plot_frame.sample(sample_size, random_state=RANDOM_STATE)

    n_cols = 3
    n_rows = int(np.ceil(len(available_columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.8 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for axis, column in zip(axes, available_columns):
        sns.histplot(data=plot_frame, x=column, hue="target", bins=35, stat="density", common_norm=False, ax=axis)
        axis.set_title(column)
        axis.set_xlabel("")

    for axis in axes[len(available_columns):]:
        axis.set_visible(False)

    fig.suptitle("Numeric Distributions by Target", y=1.01)
    plt.tight_layout()
    plt.show()


def plot_correlation_heatmap(X: pd.DataFrame, y: pd.Series, columns: Sequence[str]) -> pd.DataFrame:
    """Plot Spearman correlations for engineered numeric features plus the target."""
    engineered = add_bank_features(X)
    numeric = engineered.loc[:, columns].apply(lambda series: pd.to_numeric(series, errors="coerce"))
    numeric = numeric.assign(target=y.to_numpy())
    corr = numeric.corr(method="spearman")

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, cmap="vlag", center=0, square=True, linewidths=0.4, ax=ax)
    ax.set_title("Spearman Correlation: Engineered Numeric Features")
    plt.tight_layout()
    plt.show()
    return corr


def plot_categorical_target_rates(
    X: pd.DataFrame,
    y: pd.Series,
    columns: Sequence[str],
    min_count: int = 100,
    top_n: int = 12,
) -> None:
    """Plot target rates for selected categorical columns."""
    n_cols = 2
    n_rows = int(np.ceil(len(columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4.5 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for axis, column in zip(axes, columns):
        rates = target_rate_by_category(X, y, column, min_count=min_count).head(top_n)
        sns.barplot(data=rates, x="positive_rate", y=column, color="#4C78A8", ax=axis)
        axis.set_title(f"Positive rate by {column}")
        axis.set_xlabel("Positive-class rate")
        axis.set_ylabel("")
        axis.set_xlim(0, max(0.05, rates["positive_rate"].max() * 1.15) if not rates.empty else 0.05)

    for axis in axes[len(columns):]:
        axis.set_visible(False)

    plt.tight_layout()
    plt.show()


## Dataset Overview

Start with shape, duplicates, target balance, missingness, and type stability. The target table shows both the binary modeling label and the original 1/2 label.


In [ ]:
display(summarize_frames(X_train, y_train, X_test))
display(target_balance_table(y_train))
plot_target_balance(y_train)


In [ ]:
missing_report = missingness_table({"train": X_train, "test": X_test})
display(missing_report.query("missing_count > 0"))
plot_missingness(combined_features)


In [ ]:
display(numeric_profile(combined_features, RAW_NUMERIC_FEATURES))
display(categorical_profile(combined_features, RAW_CATEGORICAL_FEATURES))


## Target-Aware EDA

The following views use the training labels only. `duration` is expected to be very predictive in this dataset family, but it can be unavailable before a call is completed in real deployment, so it should be treated as a business-context decision rather than a blind feature choice.


In [ ]:
plot_numeric_distributions(
    X_train,
    y_train,
    columns=["age", "balance", "duration", "campaign", "pdays_clean", "previous", "balance_signed_log1p", "duration_log1p"],
)


In [ ]:
correlation_matrix = plot_correlation_heatmap(X_train, y_train, ENGINEERED_NUMERIC_FEATURES)
display(correlation_matrix["target"].sort_values(ascending=False).to_frame("spearman_with_target"))


In [ ]:
plot_categorical_target_rates(
    X_train,
    y_train,
    columns=["poutcome", "contact", "month", "housing", "loan", "job"],
    min_count=100,
)


## Preprocessing Design

Two preprocessing families are defined:

- Linear models use imputation, Yeo-Johnson transforms, scaling, and one-hot encoding.
- Tree and boosting models use median imputation plus ordinal encoding to keep the representation compact.

Both families include the same semantic feature engineering step, so training and inference follow the same path.


In [ ]:
class BankFeatureEngineer(BaseEstimator, TransformerMixin):
    """Scikit-learn transformer wrapper around add_bank_features."""

    def __init__(self, month_map: Optional[Mapping[str, int]] = None):
        self.month_map = month_map

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.feature_names_in_ = np.asarray(EXPECTED_FEATURE_COLUMNS, dtype=object)
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            frame = X.copy()
        else:
            check_is_fitted(self, "feature_names_in_")
            frame = pd.DataFrame(X, columns=self.feature_names_in_)
        return add_bank_features(frame, month_map=self.month_map or MONTH_TO_NUMBER)


def make_one_hot_encoder() -> OneHotEncoder:
    """Create a dense one-hot encoder across supported scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_ordinal_encoder() -> OrdinalEncoder:
    """Create an ordinal encoder that tolerates unseen categories."""
    try:
        return OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
    except TypeError:
        return OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)


def build_linear_preprocessor() -> Pipeline:
    """Preprocessor for linear models and interaction-heavy models."""
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("yeo_johnson", PowerTransformer(method="yeo-johnson", standardize=False)),
            ("scaler", RobustScaler()),
        ]
    )
    binary_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    column_transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, ENGINEERED_NUMERIC_FEATURES),
            ("bin", binary_pipeline, ENGINEERED_BINARY_FEATURES),
            ("cat", categorical_pipeline, ENGINEERED_CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    return Pipeline(steps=[("feature_engineering", BankFeatureEngineer()), ("columns", column_transformer)])


def build_tree_preprocessor() -> Pipeline:
    """Compact preprocessor for tree, forest, and boosting models."""
    numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    binary_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", make_ordinal_encoder()),
        ]
    )
    column_transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, ENGINEERED_NUMERIC_FEATURES),
            ("bin", binary_pipeline, ENGINEERED_BINARY_FEATURES),
            ("cat", categorical_pipeline, ENGINEERED_CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    return Pipeline(steps=[("feature_engineering", BankFeatureEngineer()), ("columns", column_transformer)])


def build_cluster_preprocessor() -> Pipeline:
    """Scaled dense representation for assigning rows to model-expert clusters."""
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]
    )
    binary_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    column_transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, ENGINEERED_NUMERIC_FEATURES),
            ("bin", binary_pipeline, ENGINEERED_BINARY_FEATURES),
            ("cat", categorical_pipeline, ENGINEERED_CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    return Pipeline(steps=[("feature_engineering", BankFeatureEngineer()), ("columns", column_transformer)])


In [ ]:
linear_preprocessor = build_linear_preprocessor()
tree_preprocessor = build_tree_preprocessor()
cluster_preprocessor = build_cluster_preprocessor()

preprocessor_blueprint = pd.DataFrame(
    [
        {"name": "linear_preprocessor", "purpose": "regularized linear and interaction models", "status": "created_not_fitted"},
        {"name": "tree_preprocessor", "purpose": "tree, forest, and boosting experts", "status": "created_not_fitted"},
        {"name": "cluster_preprocessor", "purpose": "row cluster assignment for expert routing", "status": "created_not_fitted"},
    ]
)
display(preprocessor_blueprint)


## Expert Model Registry

The registry builds unfitted model templates. XGBoost, LightGBM, CatBoost, and TabPFN are included only when their packages and runtime requirements are available. The notebook creates these objects but does not call `.fit()`.


In [ ]:
class CrossGroupInteractions(BaseEstimator, TransformerMixin):
    """Add interactions between dense numeric/binary features and one-hot categorical features."""

    def __init__(self, left_block_size: int):
        self.left_block_size = left_block_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        array = np.asarray(X)
        left = array[:, : self.left_block_size]
        right = array[:, self.left_block_size :]
        if right.shape[1] == 0:
            return array
        cross = np.einsum("ij,ik->ijk", left, right).reshape(array.shape[0], -1)
        return np.hstack([array, cross])


def compute_scale_pos_weight(y: pd.Series) -> float:
    """Return negative/positive class ratio for weighted boosting losses."""
    counts = pd.Series(y).value_counts()
    positives = counts.get(POSITIVE_CLASS, 0)
    negatives = len(y) - positives
    if positives == 0:
        raise ValueError("Cannot compute scale_pos_weight without positive examples.")
    return float(negatives / positives)


def make_hist_gradient_boosting_classifier() -> HistGradientBoostingClassifier:
    """Construct HistGradientBoostingClassifier across scikit-learn versions."""
    base_params = {
        "learning_rate": 0.06,
        "max_iter": 350,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.05,
        "random_state": RANDOM_STATE,
    }
    try:
        return HistGradientBoostingClassifier(class_weight="balanced", **base_params)
    except TypeError:
        return HistGradientBoostingClassifier(**base_params)


def build_expert_registry(scale_pos_weight: float) -> Dict[str, BaseEstimator]:
    """Build unfitted expert model templates."""
    left_block_size = len(ENGINEERED_NUMERIC_FEATURES) + len(ENGINEERED_BINARY_FEATURES)
    registry: Dict[str, BaseEstimator] = {
        "logistic_regression": make_pipeline(
            build_linear_preprocessor(),
            LogisticRegression(class_weight="balanced", max_iter=1500, random_state=RANDOM_STATE),
        ),
        "elastic_net_interactions": Pipeline(
            steps=[
                ("preprocessor", build_linear_preprocessor()),
                ("cross_group_interactions", CrossGroupInteractions(left_block_size=left_block_size)),
                (
                    "classifier",
                    LogisticRegression(
                        penalty="elasticnet",
                        solver="saga",
                        l1_ratio=0.25,
                        class_weight="balanced",
                        max_iter=2500,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "random_forest": make_pipeline(
            build_tree_preprocessor(),
            RandomForestClassifier(
                n_estimators=450,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
        "extra_trees": make_pipeline(
            build_tree_preprocessor(),
            ExtraTreesClassifier(
                n_estimators=500,
                min_samples_leaf=2,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
        "hist_gradient_boosting": make_pipeline(
            build_tree_preprocessor(),
            make_hist_gradient_boosting_classifier(),
        ),
    }

    if xgb is not None:
        registry["xgboost"] = make_pipeline(
            build_tree_preprocessor(),
            xgb.XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                n_estimators=650,
                learning_rate=0.04,
                max_depth=4,
                min_child_weight=8,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=10.0,
                reg_alpha=0.2,
                scale_pos_weight=scale_pos_weight,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        )

    if lgb is not None:
        registry["lightgbm"] = make_pipeline(
            build_tree_preprocessor(),
            lgb.LGBMClassifier(
                objective="binary",
                n_estimators=650,
                learning_rate=0.04,
                num_leaves=31,
                min_child_samples=40,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=8.0,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
                verbose=-1,
            ),
        )

    if CatBoostClassifier is not None:
        registry["catboost"] = make_pipeline(
            build_tree_preprocessor(),
            CatBoostClassifier(
                loss_function="Logloss",
                eval_metric="BalancedAccuracy",
                iterations=650,
                learning_rate=0.04,
                depth=5,
                l2_leaf_reg=8.0,
                auto_class_weights="Balanced",
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
            ),
        )

    if tp is not None and os.environ.get("TABPFN_TOKEN"):
        registry["tabpfn"] = make_pipeline(
            build_tree_preprocessor(),
            tp.TabPFNClassifier(device="cpu", n_estimators=4, random_state=RANDOM_STATE),
        )

    return registry


def summarize_expert_registry(registry: Mapping[str, BaseEstimator]) -> pd.DataFrame:
    """Return a readable summary of unfitted expert templates."""
    rows = []
    for name, estimator in registry.items():
        if isinstance(estimator, Pipeline):
            steps = " -> ".join(step_name for step_name, _ in estimator.steps)
            final_estimator = estimator.steps[-1][1].__class__.__name__
        else:
            steps = estimator.__class__.__name__
            final_estimator = estimator.__class__.__name__
        rows.append({"expert": name, "final_estimator": final_estimator, "pipeline_steps": steps, "status": "created_not_fitted"})
    return pd.DataFrame(rows).sort_values("expert")


## Clustered Expert Model

The clustered model is a mixture-of-experts scaffold. A clusterer routes similar rows to expert-specific models, while a fallback model handles small or unexpected clusters. This is defined as reusable code but remains unfitted in this notebook.


In [ ]:
def slice_rows(data, row_index: np.ndarray):
    """Slice pandas or numpy-like data while preserving the original container when possible."""
    if hasattr(data, "iloc"):
        return data.iloc[row_index]
    return data[row_index]


def aligned_predict_proba(model: BaseEstimator, X, classes: np.ndarray) -> np.ndarray:
    """Return probabilities aligned to a known class order."""
    if hasattr(model, "predict_proba"):
        raw_proba = model.predict_proba(X)
        model_classes = np.asarray(getattr(model, "classes_", classes))
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        if np.ndim(scores) == 1:
            positive = 1 / (1 + np.exp(-scores))
            raw_proba = np.column_stack([1 - positive, positive])
            model_classes = np.asarray([classes[0], classes[-1]])
        else:
            shifted = scores - scores.max(axis=1, keepdims=True)
            raw_proba = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
            model_classes = classes
    else:
        predictions = model.predict(X)
        raw_proba = np.zeros((len(predictions), len(classes)))
        model_classes = classes
        for row_index, prediction in enumerate(predictions):
            raw_proba[row_index, np.where(classes == prediction)[0][0]] = 1.0

    aligned = np.zeros((raw_proba.shape[0], len(classes)))
    for source_index, class_label in enumerate(model_classes):
        target_index = np.where(classes == class_label)[0]
        if len(target_index):
            aligned[:, target_index[0]] = raw_proba[:, source_index]
    return aligned


class ClusteredExpertClassifier(BaseEstimator, ClassifierMixin):
    """Route rows through a fitted clusterer and train one expert per cluster when fit is called."""

    def __init__(
        self,
        cluster_preprocessor: BaseEstimator,
        clusterer: BaseEstimator,
        expert_templates: Mapping[str, BaseEstimator],
        cluster_expert_map: Optional[Mapping[int, str]] = None,
        fallback_expert_name: str = "random_forest",
        min_cluster_size: int = MIN_CLUSTER_SIZE,
    ):
        self.cluster_preprocessor = cluster_preprocessor
        self.clusterer = clusterer
        self.expert_templates = expert_templates
        self.cluster_expert_map = cluster_expert_map
        self.fallback_expert_name = fallback_expert_name
        self.min_cluster_size = min_cluster_size

    def fit(self, X, y):
        if not self.expert_templates:
            raise ValueError("expert_templates must contain at least one estimator.")
        if self.fallback_expert_name not in self.expert_templates:
            raise ValueError(f"fallback_expert_name={self.fallback_expert_name!r} is not in expert_templates.")

        y_array = np.asarray(y)
        self.classes_ = np.sort(np.unique(y_array))
        self.cluster_preprocessor_ = clone(self.cluster_preprocessor)
        self.clusterer_ = clone(self.clusterer)
        self.fallback_model_ = clone(self.expert_templates[self.fallback_expert_name]).fit(X, y_array)

        cluster_matrix = self.cluster_preprocessor_.fit_transform(X, y_array)
        cluster_labels = self.clusterer_.fit_predict(cluster_matrix)
        self.cluster_models_ = {}
        self.cluster_summary_ = []

        cluster_map = dict(self.cluster_expert_map or {})
        for cluster_id in sorted(np.unique(cluster_labels)):
            row_index = np.where(cluster_labels == cluster_id)[0]
            expert_name = cluster_map.get(int(cluster_id), self.fallback_expert_name)
            if expert_name not in self.expert_templates:
                expert_name = self.fallback_expert_name

            if len(row_index) < self.min_cluster_size or len(np.unique(y_array[row_index])) < 2:
                model = DummyClassifier(strategy="most_frequent")
                selected_expert = "dummy_most_frequent"
            else:
                model = clone(self.expert_templates[expert_name])
                selected_expert = expert_name

            model.fit(slice_rows(X, row_index), y_array[row_index])
            self.cluster_models_[int(cluster_id)] = model
            self.cluster_summary_.append(
                {
                    "cluster": int(cluster_id),
                    "rows": int(len(row_index)),
                    "positive_rate": float(y_array[row_index].mean()),
                    "expert": selected_expert,
                }
            )

        return self

    def predict_proba(self, X):
        check_is_fitted(self, ["cluster_preprocessor_", "clusterer_", "cluster_models_", "classes_"])
        cluster_matrix = self.cluster_preprocessor_.transform(X)
        cluster_labels = self.clusterer_.predict(cluster_matrix)
        probabilities = np.zeros((len(cluster_labels), len(self.classes_)))

        for cluster_id in np.unique(cluster_labels):
            row_index = np.where(cluster_labels == cluster_id)[0]
            model = self.cluster_models_.get(int(cluster_id), self.fallback_model_)
            probabilities[row_index] = aligned_predict_proba(model, slice_rows(X, row_index), self.classes_)

        return probabilities

    def predict(self, X):
        probabilities = self.predict_proba(X)
        return self.classes_[np.argmax(probabilities, axis=1)]

    def cluster_summary(self) -> pd.DataFrame:
        check_is_fitted(self, "cluster_summary_")
        return pd.DataFrame(self.cluster_summary_).sort_values("cluster")


def build_cluster_expert_map(available_experts: Sequence[str], n_clusters: int = N_MODEL_CLUSTERS) -> Dict[int, str]:
    """Assign preferred experts to clusters based on installed packages."""
    preferred_order = ["xgboost", "lightgbm", "catboost", "random_forest", "extra_trees", "hist_gradient_boosting"]
    usable = [expert for expert in preferred_order if expert in available_experts]
    if not usable:
        usable = list(available_experts)
    return {cluster_id: usable[cluster_id % len(usable)] for cluster_id in range(n_clusters)}


def build_clustered_expert_model(registry: Mapping[str, BaseEstimator]) -> ClusteredExpertClassifier:
    """Create an unfitted clustered expert classifier."""
    if not registry:
        raise ValueError("Cannot build a clustered model without expert templates.")
    fallback = "random_forest" if "random_forest" in registry else next(iter(registry))
    cluster_map = build_cluster_expert_map(list(registry), n_clusters=N_MODEL_CLUSTERS)
    return ClusteredExpertClassifier(
        cluster_preprocessor=build_cluster_preprocessor(),
        clusterer=KMeans(n_clusters=N_MODEL_CLUSTERS, random_state=RANDOM_STATE, n_init=10),
        expert_templates=registry,
        cluster_expert_map=cluster_map,
        fallback_expert_name=fallback,
        min_cluster_size=MIN_CLUSTER_SIZE,
    )


def summarize_cluster_plan(model: ClusteredExpertClassifier) -> pd.DataFrame:
    """Summarize the planned cluster-to-expert routing before training."""
    cluster_map = dict(model.cluster_expert_map or {})
    return pd.DataFrame(
        [
            {"cluster": cluster_id, "planned_expert": expert_name, "status": "created_not_fitted"}
            for cluster_id, expert_name in sorted(cluster_map.items())
        ]
    )


In [ ]:
scale_pos_weight = compute_scale_pos_weight(y_train)
expert_registry = build_expert_registry(scale_pos_weight=scale_pos_weight)
clustered_expert_model = build_clustered_expert_model(expert_registry)

display(pd.DataFrame({"metric": ["scale_pos_weight"], "value": [scale_pos_weight]}))
display(summarize_expert_registry(expert_registry))
display(summarize_cluster_plan(clustered_expert_model))
print(f"Training disabled: RUN_TRAINING={RUN_TRAINING}")


## Training Entry Points (Not Executed)

These functions are here so the notebook is ready for a later modeling run. They deliberately refuse to run while `RUN_TRAINING` is `False`.


In [ ]:
def require_training_enabled() -> None:
    """Stop accidental model fitting from this notebook."""
    if not RUN_TRAINING:
        raise RuntimeError("Training is disabled. Set RUN_TRAINING = True before fitting or evaluating models.")


def make_cv_strategy(y: pd.Series, requested_splits: int = N_CV_SPLITS) -> StratifiedKFold:
    """Create a stratified CV splitter that respects the smallest class count."""
    min_class_count = int(pd.Series(y).value_counts().min())
    if min_class_count < 2:
        raise ValueError("Stratified CV requires at least two examples in each class.")
    n_splits = min(requested_splits, min_class_count)
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)


def cross_validate_expert_registry(
    registry: Mapping[str, BaseEstimator],
    X: pd.DataFrame,
    y: pd.Series,
    expert_names: Optional[Sequence[str]] = None,
    scoring: str = PRIMARY_SCORING,
) -> pd.DataFrame:
    """Cross-validate selected expert templates. Not called unless training is enabled."""
    require_training_enabled()
    names = list(expert_names or registry.keys())
    cv = make_cv_strategy(y)
    rows = []

    for name in names:
        estimator = clone(registry[name])
        scores = cross_validate(estimator, X, y, scoring=scoring, cv=cv, n_jobs=-1, return_train_score=True)
        rows.append(
            {
                "expert": name,
                "mean_train_score": scores["train_score"].mean(),
                "mean_validation_score": scores["test_score"].mean(),
                "std_validation_score": scores["test_score"].std(),
            }
        )

    return pd.DataFrame(rows).sort_values("mean_validation_score", ascending=False)


def fit_expert(registry: Mapping[str, BaseEstimator], expert_name: str, X: pd.DataFrame, y: pd.Series) -> BaseEstimator:
    """Fit one named expert. Not called unless training is enabled."""
    require_training_enabled()
    if expert_name not in registry:
        raise KeyError(f"Unknown expert: {expert_name}")
    return clone(registry[expert_name]).fit(X, y)


def fit_clustered_model(model: ClusteredExpertClassifier, X: pd.DataFrame, y: pd.Series) -> ClusteredExpertClassifier:
    """Fit the clustered expert classifier. Not called unless training is enabled."""
    require_training_enabled()
    return clone(model).fit(X, y)


def evaluate_fitted_model(model: BaseEstimator, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """Evaluate a fitted binary classifier on labeled data."""
    predictions = model.predict(X)
    probabilities = aligned_predict_proba(model, X, classes=np.asarray([0, 1]))[:, 1]
    return pd.DataFrame(
        [
            {
                "balanced_accuracy": balanced_accuracy_score(y, predictions),
                "log_loss": log_loss(y, probabilities, labels=[0, 1]),
                "positive_prediction_rate": float(np.mean(predictions)),
            }
        ]
    )


def make_prediction_frame(model: BaseEstimator, X: pd.DataFrame) -> pd.DataFrame:
    """Create a prediction frame using the original 1/2 target coding."""
    predictions = pd.Series(model.predict(X), name="prediction_binary")
    prediction_proba = aligned_predict_proba(model, X, classes=np.asarray([0, 1]))[:, 1]
    return pd.DataFrame(
        {
            "prediction": predictions.map(BINARY_TO_TARGET).astype(int),
            "positive_probability": prediction_proba,
        }
    )


In [ ]:
if RUN_TRAINING:
    cv_summary = cross_validate_expert_registry(expert_registry, X_train, y_train)
    display(cv_summary)

    fitted_clustered_model = fit_clustered_model(clustered_expert_model, X_train, y_train)
    display(fitted_clustered_model.cluster_summary())
else:
    print("No model fitting, tuning, cross-validation, or test prediction was run.")
    print("Set RUN_TRAINING = True in the configuration cell when you intentionally want to train.")


## Current Project State

The notebook now separates data checks, EDA, feature engineering, preprocessing, expert registry construction, clustered model design, and future training entry points. The executable path stops before model training, so it is safe to run for inspection and project review.
